In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import random

print("Starting Model Evaluation Phase")

# --- Step 1: Load All Necessary Data and Models ---
# This step is similar to Phase 3, loading all our assets.
try:
    user_profile_df = pd.read_csv('../dataset/processed_user_profiles.csv')
    user_skill_matrix = pd.read_csv('../dataset/processed_user_skills.csv')
    encoded_profiles = joblib.load('../models/content_based_encoded_profiles.joblib')
    skill_similarity_df = joblib.load('../models/collaborative_skill_similarity.joblib')
    print("All data and models loaded successfully.")
except FileNotFoundError:
    print("Error: Ensure all models from Phase 2 are saved.")
    exit()

# --- Step 2: Copy Over the Recommendation Functions ---
# For self-containment, we'll have the functions directly in this notebook.

def recommend_by_profile(user_id, all_encoded_profiles, user_skill_matrix, top_n_users=50, top_n_skills=10):
    target_user_vector = all_encoded_profiles[user_id]
    similarity_scores = cosine_similarity(target_user_vector, all_encoded_profiles)[0]
    similar_user_indices = np.argsort(similarity_scores)[-top_n_users-1:-1][::-1]
    similar_users_skills = user_skill_matrix.iloc[similar_user_indices]
    skill_popularity = similar_users_skills.drop(columns=['user_id']).sum().sort_values(ascending=False)
    user_known_skills_series = user_skill_matrix[user_skill_matrix['user_id'] == user_id]
    if not user_known_skills_series.empty:
        user_known_skills = user_known_skills_series.drop(columns=['user_id'])
        skills_to_remove = user_known_skills.columns[user_known_skills.iloc[0] == 1]
        final_recommendations = skill_popularity.drop(skills_to_remove, errors='ignore')
    else:
        final_recommendations = skill_popularity
    return final_recommendations.head(top_n_skills)

def recommend_by_skills(known_skills, skill_similarity_df, top_n_skills=10):
    all_recommendations = pd.Series(dtype=float)
    for skill in known_skills:
        if skill in skill_similarity_df.columns:
            recommendations = skill_similarity_df[skill].sort_values(ascending=False)
            all_recommendations = pd.concat([all_recommendations, recommendations])
    final_recommendations = all_recommendations.groupby(all_recommendations.index).sum()
    if known_skills:
        final_recommendations = final_recommendations.drop(known_skills, errors='ignore')
    return final_recommendations.sort_values(ascending=False).head(top_n_skills)

# --- Step 3: Create the Test Set ---
# We will "hide" some skills from a sample of users.

def create_test_set(skill_matrix, num_users=1000, skills_to_hide=2):
    test_set = []
    # Filter for users who have a reasonable number of skills to test on
    eligible_users = skill_matrix[skill_matrix.drop('user_id', axis=1).sum(axis=1) > skills_to_hide * 2]
    
    if len(eligible_users) < num_users:
        print(f"Warning: Only {len(eligible_users)} eligible users found. Using all of them for the test set.")
        num_users = len(eligible_users)

    sample_users = eligible_users.sample(n=num_users, random_state=42)

    for index, row in sample_users.iterrows():
        user_id = row['user_id']
        known_skills_series = row.drop('user_id')
        known_skills = known_skills_series[known_skills_series == 1].index.tolist()
        
        # Randomly select skills to hide
        hidden_skills = random.sample(known_skills, skills_to_hide)
        
        # The remaining skills are the input for the model
        input_skills = [skill for skill in known_skills if skill not in hidden_skills]
        
        test_set.append({
            'user_id': user_id,
            'input_skills': input_skills,
            'hidden_skills': hidden_skills
        })
    return test_set

print("\nCreating test set...")
test_set = create_test_set(user_skill_matrix, num_users=1000, skills_to_hide=2)
print(f"Test set created with {len(test_set)} users.")
print("Example test case:", test_set[0])

# --- Step 4: Run the Evaluation Loop ---
K = 10  # We will evaluate Precision@10 and Recall@10
results = []

for user_data in tqdm(test_set, desc="Evaluating models"):
    user_id = user_data['user_id']
    input_skills = user_data['input_skills']
    hidden_skills = user_data['hidden_skills'] # This is our ground truth

    # --- Get recommendations from each model ---
    
    # 1. Content-Based
    cb_recs = recommend_by_profile(user_id, encoded_profiles, user_skill_matrix, top_n_skills=K)
    cb_recs_list = cb_recs.index.tolist()
    
    # 2. Collaborative-Based
    # For this model, the input is the list of skills, not the user_id
    collab_recs = recommend_by_skills(input_skills, skill_similarity_df, top_n_skills=K)
    collab_recs_list = collab_recs.index.tolist()
    
    # --- Calculate metrics for each model ---
    def calculate_metrics(recs_list, ground_truth, k_val):
        hits = len(set(recs_list) & set(ground_truth))
        precision_at_k = hits / k_val
        recall_at_k = hits / len(ground_truth)
        return precision_at_k, recall_at_k

    cb_precision, cb_recall = calculate_metrics(cb_recs_list, hidden_skills, K)
    collab_precision, collab_recall = calculate_metrics(collab_recs_list, hidden_skills, K)
    
    results.append({
        'user_id': user_id,
        'cb_precision': cb_precision,
        'cb_recall': cb_recall,
        'collab_precision': collab_precision,
        'collab_recall': collab_recall
    })

# --- Step 5: Aggregate and Display Results ---
results_df = pd.DataFrame(results)

print("\n--- EVALUATION RESULTS ---")
print(f"Metrics calculated for K={K} over {len(results_df)} users.\n")

# Calculate mean scores
mean_results = results_df.mean().drop('user_id')

# Format as percentage for better readability
formatted_results = (mean_results * 100).round(2).astype(str) + '%'

print("Average Precision@K and Recall@K:\n")
print(formatted_results)

Starting Model Evaluation Phase
All data and models loaded successfully.

Creating test set...
Test set created with 1000 users.
Example test case: {'user_id': np.int64(8670), 'input_skills': [' Seaborn ', '  TensorFlow ', 'Udemy'], 'hidden_skills': ['Linear or Logistic Regression', ' Matplotlib ']}


Evaluating models: 100%|██████████| 1000/1000 [00:13<00:00, 73.71it/s]


--- EVALUATION RESULTS ---
Metrics calculated for K=10 over 1000 users.

Average Precision@K and Recall@K:

cb_precision          0.0%
cb_recall             0.0%
collab_precision    18.32%
collab_recall        91.6%
dtype: object
